## AF-cluster (in Colab form!)

Last updated H Wayment-Steele, Jan 2023

### To cluster (subsample) a big MSA:

1. Upload the big MSA.

2. Run the "**Cluster MSA**" cell, changing `EX` to your desired keyword and the name of your input a3m. MSAs will be written to "subsampled_MSAs".

3. Uncomment the correct path (containing `/content/subsampled_MSAs` ) in the "**Run Models**" cell, and run it.

4. Output PDBs will be saved in `output`.

### If you have an existing set of subsampled MSAs to run:
1. Upload a directory of MSA files to run.

2. Change the `AFcluster/data..\*a3m` path in the "**Run Models**" cell and run it.

3. Output PDBs will be saved in `output`.

In [ ]:
#@markdown ## Setup dependencies
%%bash

if [ ! -d params ]; then
  mkdir params
  curl -fsSL https://storage.googleapis.com/alphafold/alphafold_params_2021-07-14.tar | tar x -C params
fi

if [ ! -d AFcluster ];then
  git clone https://github.com/HWaymentSteele/AFcluster.git
fi

if [ ! -d alphafold ]; then
  git clone https://github.com/deepmind/alphafold.git
  ! pip -q install ml-collections dm-haiku biopython
fi

if [ ! -d output ]; then
  mkdir output
fi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 371.7/371.7 kB 11.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 26.2 MB/s eta 0:00:00


Cloning into 'AFcluster'...
Updating files: 100% (13618/13618), done.
Cloning into 'alphafold'...


In [ ]:
#To solve for package dependencies I need to install biopython
!pip install biopython==1.81

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 20.1 MB/s eta 0:00:00
  Attempting uninstall: biopython
    Found existing installation: biopython 1.83
    Uninstalling biopython-1.83:
      Successfully uninstalled biopython-1.83


In [ ]:
#@markdown ## Define functions to run AlphaFold

import sys
import os
import argparse
import hashlib
import jax
import jax.numpy as jnp
import numpy as np
import re
import subprocess
from glob import glob
import pandas as pd

sys.path.append('alphafold')

from alphafold.model import model, config, data
from alphafold.data import parsers, pipeline
from alphafold.common import protein

"""
Create an AlphaFold model runner
name -- The name of the model to get the parameters from. Options: model_[1-5]
"""

def make_model_runner(model_num=3, recycles=1, deterministic=True):
  model_name = 'model_%d_ptm' % model_num
  cfg = config.model_config(model_name)

  cfg.data.common.num_recycle = recycles
  cfg.model.num_recycle = recycles
  cfg.data.eval.num_ensemble = 1
  if deterministic:
    cfg.data.eval.masked_msa_replace_fraction = 0.0
    cfg.model.global_config.deterministic = True
  params = data.get_model_haiku_params(model_name, '.')

  return model.RunModel(cfg, params)

def make_processed_feature_dict(runner, a3m_file, name="test", seed=0):
  feature_dict = {}

  # assumes sequence is first entry in msa
  with open(a3m_file,'r') as msa_fil:
    sequence = msa_fil.read().splitlines()[1].strip()

  feature_dict.update(pipeline.make_sequence_features(sequence, name, len(sequence)))

  with open(a3m_file,'r') as msa_fil:
    msa = pipeline.parsers.parse_a3m(msa_fil.read())

  feature_dict.update(pipeline.make_msa_features([msa]))
  processed_feature_dict = runner.process_features(feature_dict, random_seed=seed)

  return processed_feature_dict

"""
Package AlphaFold's output into an easy-to-use dictionary
prediction_result - output from running AlphaFold on an input dictionary
processed_feature_dict -- The dictionary passed to AlphaFold as input. Returned by `make_processed_feature_dict`.
"""
def parse_results(prediction_result, processed_feature_dict):
  b_factors = prediction_result['plddt'][:,None] * prediction_result['structure_module']['final_atom_mask']
  dist_bins = jax.numpy.append(0,prediction_result["distogram"]["bin_edges"])
  dist_mtx = dist_bins[prediction_result["distogram"]["logits"].argmax(-1)]
  contact_mtx = jax.nn.softmax(prediction_result["distogram"]["logits"])[:,:,dist_bins < 8].sum(-1)

  out = {"unrelaxed_protein": protein.from_prediction(processed_feature_dict, prediction_result, b_factors=b_factors),
        "plddt": prediction_result['plddt'],
        "pLDDT": prediction_result['plddt'].mean(),
        "dists": dist_mtx,
        "adj": contact_mtx}

  out.update({"pae": prediction_result['predicted_aligned_error'],
              "pTMscore": prediction_result['ptm']})
  return out

def write_results(result, pdb_out_path):
  plddt = float(result['pLDDT'])
  ptm = float(result["pTMscore"])
  print('plddt: %.3f' % plddt)
  print('ptm  : %.3f' % ptm)

  pdb_lines = protein.to_pdb(result["unrelaxed_protein"])
  with open(pdb_out_path, 'w') as f:
    f.write(pdb_lines)

/usr/local/lib/python3.10/dist-packages/Bio/Data/SCOPData.py:18: BiopythonDeprecationWarning: The 'Bio.Data.SCOPData' module will be deprecated in a future release of Biopython in favor of 'Bio.Data.PDBData.
  warnings.warn(


# **Cluster MSA**

In [ ]:
%%bash

pip -q install polyleven
#Original python script
#python AFcluster/scripts/ClusterMSA.py EX -i AFcluster/data_sep2022/00_KaiB/2QKEE_colabfold.a3m -o subsampled_MSAs
#Python script including PCA /content/AFcluster/data_sep2022/CaM123/CaM123_a8a54.a3m
python AFcluster/scripts/ClusterMSA.py EX -i AFcluster/data_sep2022/CaM123/CaM123_a8a54.a3m -o subsampled_MSAs --run_PCA

EX
6664 seqs removed for containing more than 25% gaps, 11628 remaining
eps	n_clusters	n_not_clustered
3.00	7	2844
3.50	8	2828
4.00	7	2798
4.50	8	2771
5.00	12	2702
5.50	15	2656
6.00	15	2597
6.50	22	2511
7.00	19	2456
7.50	18	2399
8.00	19	2328
8.50	22	2206
9.00	25	2046
9.50	30	1871
10.00	33	1651
10.50	22	1428
11.00	21	1146
11.50	7	728
12.00	2	210
12.50	2	10
13.00	1	0
Selected eps=10.00
11628 total seqs
224 clusters, 5539 of 11628 not clustered (0.48)
avg identity to query of unclustered: 0.33
avg identity to query of clustered: 0.59
writing 10 size-10 uniformly sampled clusters
writing 10 size-100 uniformly sampled clusters
Running PCA ...
Saved PCA plot to subsampled_MSAs/EX_PCA.pdf
wrote clustering data to subsampled_MSAs/EX_clustering_assignments.tsv
wrote cluster metadata to subsampled_MSAs/EX_cluster_metadata.tsv
Saved this output to EX.log


/content/AFcluster/scripts/ClusterMSA.py:19: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  tmp = df.loc[df.dbscan_label>=0][df.dbscan_label<=9]


## **MSA Transformer**

In [ ]:
#Installation needed for next step

!pip -q install biotite
!pip -q install fair-esm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 29.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 3.2 MB/s eta 0:00:00


In [ ]:
%%bash
python AFcluster/scripts/runESM.py subsampled_MSAs/*.a3m -o MSA_preds

predicting...
448.31982
wrote MSA_preds/msa_t__EX_000.npy
predicting...
267.95245
wrote MSA_preds/msa_t__EX_001.npy
predicting...
442.55426
wrote MSA_preds/msa_t__EX_002.npy
predicting...
368.6973
wrote MSA_preds/msa_t__EX_003.npy
predicting...
322.84253
wrote MSA_preds/msa_t__EX_004.npy
predicting...
296.51468
wrote MSA_preds/msa_t__EX_005.npy
predicting...
294.62762
wrote MSA_preds/msa_t__EX_006.npy
predicting...
411.8498
wrote MSA_preds/msa_t__EX_007.npy
predicting...
212.25107
wrote MSA_preds/msa_t__EX_008.npy
predicting...
287.0489
wrote MSA_preds/msa_t__EX_009.npy
predicting...
351.07364
wrote MSA_preds/msa_t__EX_010.npy
predicting...
470.5122
wrote MSA_preds/msa_t__EX_011.npy
predicting...
321.96545
wrote MSA_preds/msa_t__EX_012.npy
predicting...
297.58862
wrote MSA_preds/msa_t__EX_013.npy
predicting...
351.08258
wrote MSA_preds/msa_t__EX_014.npy
predicting...
328.35712
wrote MSA_preds/msa_t__EX_015.npy
predicting...
383.04556
wrote MSA_preds/msa_t__EX_016.npy
predicting...
397.

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm_msa1b_t12_100M_UR50S.pt" to /root/.cache/torch/hub/checkpoints/esm_msa1b_t12_100M_UR50S.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm_msa1b_t12_100M_UR50S-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm_msa1b_t12_100M_UR50S-contact-regression.pt


In [ ]:
n_recycles = 3
model_number = 3
seed=0
#name='KaiB_TE'
name='CaM123_TE'

runner = make_model_runner(model_num=model_number, recycles=n_recycles)

#subsampled_msas = glob('AFcluster/data_sep2022/00_KaiB/kaib_dbscan_msas/*a3m')
subsampled_msas = glob('subsampled_MSAs/*a3m')

for fil in subsampled_msas:
  print(fil)
  features = make_processed_feature_dict(runner, fil, seed=seed)
  result = parse_results(runner.predict(features, random_seed=seed), features)
  write_results(result, 'output/' + os.path.basename(fil).replace('.a3m','.pdb'))

subsampled_MSAs/EX_110.a3m
plddt: 73.963
ptm  : 0.624
subsampled_MSAs/EX_U10-000.a3m
plddt: 85.658
ptm  : 0.788
subsampled_MSAs/EX_045.a3m
plddt: 83.035
ptm  : 0.741
subsampled_MSAs/EX_234.a3m
plddt: 66.588
ptm  : 0.563
subsampled_MSAs/EX_039.a3m
plddt: 79.396
ptm  : 0.714
subsampled_MSAs/EX_167.a3m
plddt: 83.406
ptm  : 0.778
subsampled_MSAs/EX_180.a3m
plddt: 77.080
ptm  : 0.707
subsampled_MSAs/EX_226.a3m
plddt: 77.901
ptm  : 0.654
subsampled_MSAs/EX_228.a3m
plddt: 79.851
ptm  : 0.670
subsampled_MSAs/EX_117.a3m
plddt: 81.266
ptm  : 0.707
subsampled_MSAs/EX_238.a3m
plddt: 77.612
ptm  : 0.649
subsampled_MSAs/EX_022.a3m
plddt: 76.654
ptm  : 0.675
subsampled_MSAs/EX_239.a3m
plddt: 67.508
ptm  : 0.570
subsampled_MSAs/EX_097.a3m
plddt: 78.042
ptm  : 0.656
subsampled_MSAs/EX_166.a3m
plddt: 77.291
ptm  : 0.665
subsampled_MSAs/EX_065.a3m
plddt: 81.849
ptm  : 0.727
subsampled_MSAs/EX_173.a3m
plddt: 76.864
ptm  : 0.661
subsampled_MSAs/EX_069.a3m
plddt: 77.778
ptm  : 0.648
subsampled_MSAs/EX_U10-0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# PCA embeddings

In [ ]:
!pip install polyleven




In [ ]:
%%bash
python AFcluster/scripts/get_PCA_embedding.py subsampled_MSAs/*.a3m

## **RMSD**

In [ ]:
!pip install mdtraj
!pip install pandarallel
import mdtraj as md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 14.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for mdtraj: filename=mdtraj-1.9.9-cp310-cp310-linux_x86_64.whl size=7547333 sha256=2f01ba021ab8c34fd31213bf794e5ec87e52f0b73a80564ad78c41e102c765bd
  Stored in directory: /root/.cache/pip/wheels/4b/4e/d5/22b44e04aca0780281ffb43717ebb3404bc1a77cf8f345fa73
Successfully built mdtraj
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 4.5 MB/s eta 0:00:00
  Created wheel for pandarallel: filename=pandarallel-1.6.5-py3-none-any.whl size=16673 sha256=df485bc5cb5f58f39d259b721356381a79e4783ee2e88ea12e68d8c3d08f154d
  Stored in directory: /root/.cache/pip/wheels/50/4f/1e/34e057bb868842209f1623f195b74fd7eda229308a7352d47f
Successfully built pandarallel


In [ ]:
%%bash
##python AFcluster/scripts/CalculateModelFeatures.py /content/content/output/*.pdb -o rmsds.json.zip --ref_struct CaE34_REF_1.pdb --ref_struct CaE34_REF_2.pdb
python AFcluster/scripts/CalculateModelFeatures.py /content/output/*.pdb -o rmsds.json.zip --plot_results plot_results --ref_struct CaM34_REF_1.pdb --ref_struct CaM34_REF_2.pdb

INFO: Pandarallel will run on 4 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.
Found 490 pdbs
Wrote to rmsds.json.zip


In [ ]:
!unzip CaE234_1e9a1.result.zip

Archive:  CaE234_1e9a1.result.zip
   creating: CaE234_1e9a1/
  inflating: CaE234_1e9a1/CaE234_1e9a1_unrelaxed_rank_002_alphafold2_ptm_model_1_seed_000.pdb  
  inflating: CaE234_1e9a1/CaE234_1e9a1_unrelaxed_rank_004_alphafold2_ptm_model_4_seed_000.pdb  
  inflating: CaE234_1e9a1/CaE234_1e9a1_unrelaxed_rank_005_alphafold2_ptm_model_2_seed_000.pdb  
  inflating: CaE234_1e9a1/CaE234_1e9a1.csv  
  inflating: CaE234_1e9a1/CaE234_1e9a1_scores_rank_003_alphafold2_ptm_model_5_seed_000.json  
  inflating: CaE234_1e9a1/config.json  
 extracting: CaE234_1e9a1/CaE234_1e9a1.done.txt  
  inflating: CaE234_1e9a1/CaE234_1e9a1_unrelaxed_rank_003_alphafold2_ptm_model_5_seed_000.pdb  
  inflating: CaE234_1e9a1/CaE234_1e9a1_scores_rank_002_alphafold2_ptm_model_1_seed_000.json  
  inflating: CaE234_1e9a1/CaE234_1e9a1_scores_rank_001_alphafold2_ptm_model_3_seed_000.json  
  inflating: CaE234_1e9a1/CaE234_1e9a1_plddt.png  
  inflating: CaE234_1e9a1/CaE234_1e9a1.a3m  
  inflating: CaE234_1e9a1/cite.bibtex  
  

In [ ]:
!zip -r /content/subsamples_MSAs.zip /content/subsampled_MSAs/

  adding: content/subsampled_MSAs/ (stored 0%)
  adding: content/subsampled_MSAs/EX_144.a3m (deflated 50%)
  adding: content/subsampled_MSAs/EX_275.a3m (deflated 49%)
  adding: content/subsampled_MSAs/EX_U100-007.a3m (deflated 51%)
  adding: content/subsampled_MSAs/EX_155.a3m (deflated 44%)
  adding: content/subsampled_MSAs/EX_112.a3m (deflated 46%)
  adding: content/subsampled_MSAs/EX_242.a3m (deflated 45%)
  adding: content/subsampled_MSAs/EX_461.a3m (deflated 42%)
  adding: content/subsampled_MSAs/EX_311.a3m (deflated 42%)
  adding: content/subsampled_MSAs/EX_024.a3m (deflated 44%)
  adding: content/subsampled_MSAs/EX_006.a3m (deflated 55%)
  adding: content/subsampled_MSAs/EX_392.a3m (deflated 44%)
  adding: content/subsampled_MSAs/EX_150.a3m (deflated 54%)
  adding: content/subsampled_MSAs/EX_188.a3m (deflated 48%)
  adding: content/subsampled_MSAs/EX_044.a3m (deflated 45%)
  adding: content/subsampled_MSAs/EX_366.a3m (deflated 39%)
  adding: content/subsampled_MSAs/EX_210.a3m (de

In [ ]:
!zip -r /content/CaM34_output.zip /content/output/

  adding: content/output/ (stored 0%)
  adding: content/output/EX_110.pdb (deflated 78%)
  adding: content/output/EX_U100-009.pdb (deflated 78%)
  adding: content/output/EX_203.pdb (deflated 78%)
  adding: content/output/EX_296.pdb (deflated 78%)
  adding: content/output/EX_204.pdb (deflated 78%)
  adding: content/output/EX_049.pdb (deflated 78%)
  adding: content/output/EX_129.pdb (deflated 78%)
  adding: content/output/EX_028.pdb (deflated 78%)
  adding: content/output/EX_377.pdb (deflated 78%)
  adding: content/output/EX_237.pdb (deflated 78%)
  adding: content/output/EX_313.pdb (deflated 78%)
  adding: content/output/EX_184.pdb (deflated 78%)
  adding: content/output/EX_360.pdb (deflated 78%)
  adding: content/output/EX_461.pdb (deflated 77%)
  adding: content/output/EX_395.pdb (deflated 78%)
  adding: content/output/EX_433.pdb (deflated 78%)
  adding: content/output/EX_326.pdb (deflated 78%)
  adding: content/output/EX_235.pdb (deflated 78%)
  adding: content/output/EX_257.pdb (de

In [ ]:
!zip -r /content/CaM34_MSA_preds.zip /content/MSA_preds/

  adding: content/MSA_preds/ (stored 0%)
  adding: content/MSA_preds/msa_t__EX_054.npy (deflated 61%)
  adding: content/MSA_preds/msa_t__EX_422.npy (deflated 61%)
  adding: content/MSA_preds/msa_t__EX_367.npy (deflated 61%)
  adding: content/MSA_preds/msa_t__EX_383.npy (deflated 61%)
  adding: content/MSA_preds/msa_t__EX_150.npy (deflated 61%)
  adding: content/MSA_preds/msa_t__EX_153.npy (deflated 61%)
  adding: content/MSA_preds/msa_t__EX_429.npy (deflated 61%)
  adding: content/MSA_preds/msa_t__EX_145.npy (deflated 61%)
  adding: content/MSA_preds/msa_t__EX_399.npy (deflated 61%)
  adding: content/MSA_preds/msa_t__EX_025.npy (deflated 61%)
  adding: content/MSA_preds/msa_t__EX_405.npy (deflated 61%)
  adding: content/MSA_preds/msa_t__EX_463.npy (deflated 61%)
  adding: content/MSA_preds/msa_t__EX_076.npy (deflated 61%)
  adding: content/MSA_preds/msa_t__EX_322.npy (deflated 61%)
  adding: content/MSA_preds/msa_t__EX_004.npy (deflated 61%)
  adding: content/MSA_preds/msa_t__EX_263.np

In [ ]:
from google.colab import files
files.download('CaM34_output.zip')
files.download('CaM34_MSA_preds.zip')
files.download('rmsds.json.zip')
files.download('subsamples_MSAs.zip')
files.download('df_with_PCs.json.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
%%bash

python AFcluster/scripts/ClusterMSA.py EX -i AFcluster/data_sep2022/CaE234/CaE234_1e9a1.a3m -o subsampled_MSAs --run_PCA

Traceback (most recent call last):
  File "/content/AFcluster/scripts/ClusterMSA.py", line 6, in <module>
    from polyleven import levenshtein
ModuleNotFoundError: No module named 'polyleven'


CalledProcessError: Command 'b'\npython AFcluster/scripts/ClusterMSA.py EX -i AFcluster/data_sep2022/CaE234/CaE234_1e9a1.a3m -o subsampled_MSAs --run_PCA\n'' returned non-zero exit status 1.